# 03c — A2A walkthrough

Drive the provider’s `BandwidthProviderExecutor` in-process. No port; no httpx; no real client. We fabricate a `RequestContext` and an `EventQueue`, and capture the executor’s enqueued events.

In [ ]:
import sys, pathlib
_ROOT = pathlib.Path.cwd().resolve()
if (_ROOT / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT))
elif (_ROOT.parent / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT.parent))

from notebooks._viz import render_chat_log, render_mermaid
from shared.config import Config
from provider.agent_executor import BandwidthProviderExecutor
from provider.mcp_server import build_mcp_server
from a2a.types import Message, Part
from google.protobuf.json_format import MessageToDict, ParseDict
from google.protobuf.struct_pb2 import Struct, Value
from unittest.mock import MagicMock

PROVIDER = '0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d'
cfg = Config(provider_private_key=PROVIDER, sdn_mock=True)
mcp, _ = build_mcp_server(cfg)
executor = BandwidthProviderExecutor(mcp)
print('executor ready')

## FakeQueue + helpers

The real queue () ships events to the HTTP transport. For in-process testing we capture them in a list.

In [ ]:
class FakeQueue:
    def __init__(self): self.events = []
    async def enqueue_event(self, e): self.events.append(e)

def data_part(d):
    s = Struct(); ParseDict(d, s)
    return Part(data=Value(struct_value=s), media_type='application/json')

def make_context(payload):
    msg = Message(message_id='m1', parts=[data_part(payload)])
    ctx = MagicMock()
    ctx.message = msg
    ctx.task_id = 't1'
    ctx.context_id = 'c1'
    return ctx

def payload_of(event):
    return MessageToDict(event.artifact.parts[0].data, preserving_proto_field_name=True)

## Action 1 — 

In [ ]:
import asyncio

async def run_action(payload):
    q = FakeQueue()
    await executor.execute(make_context(payload), q)
    return q.events

events = asyncio.get_event_loop().run_until_complete(run_action({"action": "get_catalog"}))
for e in events:
    cls = type(e).__name__
    print(cls)
    if hasattr(e, 'artifact'):
        print(' ', payload_of(e))
    elif hasattr(e, 'status'):
        print(' status =', e.status.state)

## Action 2 — 

In [ ]:
events = asyncio.get_event_loop().run_until_complete(run_action({
    "action": "request_quote", "package_id": "medium",
    "consumer_address": "0x000000000000000000000000000000000000dEaD"}))
log = []
for e in events:
    if hasattr(e, 'artifact'):
        log.append({"from": "provider", "message": str(payload_of(e))})
render_chat_log(log)

## Action 3 —  (deferred)

 requires a real signed nonce against a deployed contract. We skip it here because that needs anvil. The full path is exercised in [06 — end to end](06_end_to_end.ipynb).

## Internal dispatch

In [ ]:
render_mermaid("""
graph LR
  A[action: get_catalog] --> M1[MCP get_catalog]
  B[action: request_quote] --> M2[MCP request_quote]
  C[action: activate] --> M3[MCP verify_credential_ownership]
  C --> M4[MCP allocate_bandwidth]
""")

The executor's job is just to route. Look at `provider/agent_executor.py:111-173` — `_handle_catalog`, `_handle_quote`, `_handle_activate` are each a few lines that pass-through to the MCP client. Keeping the executor thin means new actions can be added with one match arm + one `_handle_*` helper.

Next: [04a — graph state schema](04a_graph_state_schema.ipynb).